In [0]:
SECRET_SCOPE = "mercadata-adls-oauth"
    # Nome padronizado do Secret Scope utilizado pela dupla.
    # Devemos ter um scope com este mesmo nome, em seus respectivos workspaces Databricks.

client_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="client-id",
)
    # Recupera as credenciais armazenadas de forma segura no Databricks.
    # O valor real das credenciais NÃO fica escrito no notebook e, portanto, não precisa ser versionado no GitHub.


tenant_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="tenant-id",
)

client_secret = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="client-secret",
)

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)

    # Validação simples:
    # confirma apenas que os valores foram carregados.

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
    # Bibliotecas do Azure utilizadas para autenticação e acesso ao Data Lake.

storage_account_name = "internshipdatalake"
container_name = "raw"
    # Configurações compartilhadas do projeto.
    # Diferente do client_secret, estes valores representam
    # a localização do storage utilizado pelo projeto.

credential = ClientSecretCredential(tenant_id, client_id, client_secret)
    # Cria o objeto de credencial OAuth.
    # Aqui estamos dizendo ao Azure:
    # "quero me autenticar usando este tenant,
    # esta aplicação e este segredo".

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential,
)
    # Cria o cliente que representa o Storage Account.

fs_client = service_client.get_file_system_client(container_name)
    # Obtém um cliente específico para o container "raw".

print("Clientes de acesso ao ADLS preparados.")


jdbc_hostname = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_hostname"
)

jdbc_database = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_database"
)

jdbc_username = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_username"
)

jdbc_password = dbutils.secrets.get(
    scope="mercadata-sqlserver",
    key="jdbc_password"
)

print("JDBC Hostname carregado:", jdbc_hostname is not None)
print("JDBC Database carregado:", jdbc_database is not None)
print("JDBC Username carregado:", jdbc_username is not None)
print("JDBC Password carregado:", jdbc_password is not None)

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname};databaseName={jdbc_database}"

connection_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
teste_conexao = spark.read.jdbc(
    url=jdbc_url,
    table="(SELECT 1 AS conexao) AS teste",
    properties=connection_properties
)

display(teste_conexao)